# Joint mouse–human clustering pipeline

This notebook contains only the reproducible path from the two spatial-transcriptomics datasets to one joint intra/inter-species Leiden clustering. Hyperparameter searches, PCA experiments, and detailed evaluation live in separate notebooks.

The final cell saves a self-describing core run. All downstream notebooks load that directory and validate the ortholog count, matrix dimensions, species split, and gene order.


In [ ]:
from pathlib import Path
import sys


def find_pipeline_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "Cross Species Pipeline", cwd.parent, cwd.parent.parent]
    for candidate in candidates:
        if (candidate / "src" / "cross_species").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate Cross Species Pipeline/src/cross_species")


PIPELINE_ROOT = find_pipeline_root()
PROJECT_ROOT = PIPELINE_ROOT.parent
SRC_ROOT = PIPELINE_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
RESULTS_ROOT = PIPELINE_ROOT / "results"
DATA_ROOT = PROJECT_ROOT / "data"
print(f"Pipeline root: {PIPELINE_ROOT}")


## Configuration


In [ ]:
from datetime import datetime
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import smoothie

from cross_species import (
    align_matrices,
    dual_knn_graph,
    load_homology,
    plot_edges,
    profile_correlation,
    save_core_run,
    stitch_profiles,
)

smoothie.suppress_warnings()

RAW_HUMAN_PATH = DATA_ROOT / "GSM9629357_CS23_E2S1_bin50.h5ad"
RAW_MOUSE_PATH = DATA_ROOT / "E16.5_E1S1.MOSTA.h5ad"
SMOOTHED_HUMAN_PATH = DATA_ROOT / "sm_adata/human_GSM9629357_CS23_E2S1_bin50_sm.h5ad"
SMOOTHED_MOUSE_PATH = DATA_ROOT / "sm_adata/mouse_E1S1_165_bin50_sm.h5ad"
HUMAN_PCC_PATH = DATA_ROOT / "pearsonR/human_GSM9629357_CS23_E2S1_bin50_pearsonR.npy"
HUMAN_GENE_PATH = DATA_ROOT / "pearsonR/human_GSM9629357_CS23_E2S1_bin50_gene_names.npy"
MOUSE_PCC_PATH = DATA_ROOT / "pearsonR/mouse_E1S1_165_bin50_pearsonR.npy"
MOUSE_GENE_PATH = DATA_ROOT / "pearsonR/mouse_E1S1_165_bin50_gene_names.npy"
HOMOLOGY_PATH = DATA_ROOT / "HOM_AllOrganism.rpt"

RECOMPUTE_SMOOTHING = False
RECOMPUTE_WITHIN_SPECIES_PCC = False
SAVE_CORE_ARTIFACTS = True

GRAPH_PARAMETERS = {
    "k_intra": 50,
    "k_inter": 50,
    "cutoff_intra": 0.90,
    "cutoff_inter": 0.70,
}
LEIDEN_RESOLUTION = 1.0
LEIDEN_SEED = 0
CORE_RUN_DIR = RESULTS_ROOT / "core" / datetime.now().strftime("core_%Y%m%d_%H%M%S")


## Load or produce smoothed expression


In [ ]:
if RECOMPUTE_SMOOTHING:
    adata_human = ad.read_h5ad(RAW_HUMAN_PATH)
    adata_mouse = ad.read_h5ad(RAW_MOUSE_PATH)
    sc.pp.filter_genes(adata_human, min_cells=50)
    sc.pp.filter_genes(adata_mouse, min_cells=50)

    sm_adata_human = smoothie.run_parallelized_smoothing(
        adata_human,
        grid_based_or_not=False,
        gaussian_sd=30.0 * 2,
        min_spots_under_gaussian=5,
    )
    sm_adata_mouse = smoothie.run_parallelized_smoothing(
        adata_mouse,
        grid_based_or_not=False,
        gaussian_sd=30.0 / 25.0,
        min_spots_under_gaussian=5,
    )
    SMOOTHED_HUMAN_PATH.parent.mkdir(parents=True, exist_ok=True)
    sm_adata_human.write_h5ad(SMOOTHED_HUMAN_PATH)
    sm_adata_mouse.write_h5ad(SMOOTHED_MOUSE_PATH)
else:
    sm_adata_human = ad.read_h5ad(SMOOTHED_HUMAN_PATH)
    sm_adata_mouse = ad.read_h5ad(SMOOTHED_MOUSE_PATH)

print("human smoothed:", sm_adata_human.shape)
print("mouse smoothed:", sm_adata_mouse.shape)


## Within-species gene correlations


In [ ]:
if RECOMPUTE_WITHIN_SPECIES_PCC:
    pearson_human, _ = smoothie.compute_correlation_matrix(sm_adata_human.X)
    genes_human = sm_adata_human.var_names.to_numpy()
    pearson_mouse, _ = smoothie.compute_correlation_matrix(sm_adata_mouse.X)
    genes_mouse = sm_adata_mouse.var_names.to_numpy()

    HUMAN_PCC_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(HUMAN_PCC_PATH, pearson_human)
    np.save(HUMAN_GENE_PATH, genes_human)
    np.save(MOUSE_PCC_PATH, pearson_mouse)
    np.save(MOUSE_GENE_PATH, genes_mouse)
else:
    pearson_human = np.load(HUMAN_PCC_PATH, allow_pickle=False)
    genes_human = np.load(HUMAN_GENE_PATH, allow_pickle=True)
    pearson_mouse = np.load(MOUSE_PCC_PATH, allow_pickle=False)
    genes_mouse = np.load(MOUSE_GENE_PATH, allow_pickle=True)

print("human correlation matrix:", pearson_human.shape)
print("mouse correlation matrix:", pearson_mouse.shape)


## Align one-to-one orthologs


In [ ]:
homology = load_homology(HOMOLOGY_PATH)
alignment = align_matrices(
    mat_a=pearson_mouse,
    genes_a=genes_mouse,
    species_a="mouse, laboratory",
    pcc_thresh_a=0.4,
    mat_b=pearson_human,
    genes_b=genes_human,
    species_b="human",
    pcc_thresh_b=0.2,
    hom=homology,
    keep_rest=True,
    strict_multiclass=True,
)
alignment.summary()
alignment.check()
assert alignment.n_orthologs == len(alignment.pairs)


## Build ortholog-correlation profiles and second-order PCC


In [ ]:
prof = stitch_profiles(alignment)
assert prof.meta["n_ortho"] == alignment.n_orthologs

C, edge_sets = profile_correlation(
    prof,
    use_pca=False,
    return_matrix=True,
)
display(edge_sets.stats)
plot_edges(edge_sets)
plt.show()


## Build the joint graph and run Leiden


In [ ]:
graph = dual_knn_graph(
    C,
    split=prof.meta["width_a"],
    genes=prof.genes,
    species=prof.species,
    **GRAPH_PARAMETERS,
)
labels = graph.leiden(
    resolution=LEIDEN_RESOLUTION,
    seed=LEIDEN_SEED,
)
graph.summary()
display(graph.mixing(labels).head(20))
print(f"Leiden modules: {len(np.unique(labels)):,}")


## Save the core run


In [ ]:
if SAVE_CORE_ARTIFACTS:
    manifest = save_core_run(
        CORE_RUN_DIR,
        prof=prof,
        C=C,
        graph=graph,
        labels=labels,
        graph_parameters=GRAPH_PARAMETERS,
        resolution=LEIDEN_RESOLUTION,
        leiden_seed=LEIDEN_SEED,
        edge_sets=edge_sets,
        ortholog_pairs=alignment.pairs,
        input_paths={
            "smoothed_human": SMOOTHED_HUMAN_PATH,
            "smoothed_mouse": SMOOTHED_MOUSE_PATH,
            "human_pcc": HUMAN_PCC_PATH,
            "mouse_pcc": MOUSE_PCC_PATH,
            "homology": HOMOLOGY_PATH,
        },
        notes="Canonical joint mouse-human clustering run.",
    )
    display(pd.Series(manifest, name="value"))
else:
    print("SAVE_CORE_ARTIFACTS is False; no downstream artifact directory was written.")
